# Quick Maths with Matrices!
---

<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Quick-Maths-with-Matrices!" data-toc-modified-id="Quick-Maths-with-Matrices!-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Quick Maths with Matrices!</a></span><ul class="toc-item"><li><span><a href="#Import-Libraries" data-toc-modified-id="Import-Libraries-1.1"><span class="toc-item-num">1.1&nbsp;&nbsp;</span>Import Libraries</a></span></li><li><span><a href="#Test-Framework" data-toc-modified-id="Test-Framework-1.2"><span class="toc-item-num">1.2&nbsp;&nbsp;</span>Test Framework</a></span></li></ul></li><li><span><a href="#Optimizing-Matrix-Multiplications" data-toc-modified-id="Optimizing-Matrix-Multiplications-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Optimizing Matrix Multiplications</a></span><ul class="toc-item"><li><span><a href="#For-Loop" data-toc-modified-id="For-Loop-2.1"><span class="toc-item-num">2.1&nbsp;&nbsp;</span>For Loop</a></span></li><li><span><a href="#Array-Slicing" data-toc-modified-id="Array-Slicing-2.2"><span class="toc-item-num">2.2&nbsp;&nbsp;</span>Array Slicing</a></span></li><li><span><a href="#Improvement-with-array-slicing" data-toc-modified-id="Improvement-with-array-slicing-2.3"><span class="toc-item-num">2.3&nbsp;&nbsp;</span>Improvement with array slicing</a></span></li><li><span><a href="#Array-Broadcasting" data-toc-modified-id="Array-Broadcasting-2.4"><span class="toc-item-num">2.4&nbsp;&nbsp;</span>Array Broadcasting</a></span></li><li><span><a href="#Improvement-with-array-broadcasting" data-toc-modified-id="Improvement-with-array-broadcasting-2.5"><span class="toc-item-num">2.5&nbsp;&nbsp;</span>Improvement with array broadcasting</a></span></li><li><span><a href="#Einstein-Sum" data-toc-modified-id="Einstein-Sum-2.6"><span class="toc-item-num">2.6&nbsp;&nbsp;</span>Einstein Sum</a></span></li><li><span><a href="#Improvement-with-einstein-sum" data-toc-modified-id="Improvement-with-einstein-sum-2.7"><span class="toc-item-num">2.7&nbsp;&nbsp;</span>Improvement with einstein sum</a></span></li><li><span><a href="#Linear-Algebra-Libraries" data-toc-modified-id="Linear-Algebra-Libraries-2.8"><span class="toc-item-num">2.8&nbsp;&nbsp;</span>Linear Algebra Libraries</a></span></li><li><span><a href="#Improvement-with-linear-algebra-libraries" data-toc-modified-id="Improvement-with-linear-algebra-libraries-2.9"><span class="toc-item-num">2.9&nbsp;&nbsp;</span>Improvement with linear algebra libraries</a></span></li></ul></li></ul></div>

## Import Libraries

In [54]:
import torch
import timeit
import operator
from functools import partial

## Test Framework

In [55]:
def test(a, b, compare, compare_name=None):
    if compare_name is None:
        compare_name = compare.__name__
    assert compare(a, b),\
    f"{compare_name} check failed:\n{a}\n{b}"

def test_equality(a, b):
    test(a, b, operator.eq, "Equality")

def test_approximately(a, b):
    allclose = partial(torch.allclose, atol=1e-5, rtol=1e-03)
    if not isinstance(a, torch.Tensor) or not isinstance(b, torch.Tensor):
        a = torch.tensor(a)
        b = torch.tensor(b)
    test(a, b, allclose, "Approximate Equality")

In [56]:
test_equality(1e-5,1e-5)

In [57]:
test_approximately(1e-5, 1e-6)

# Optimizing Matrix Multiplications

**Test Variables**

In [67]:
A = torch.randn([10,10])
B = torch.randn([10,10])

In [69]:
(A@B).shape

torch.Size([10, 10])

## For Loop

In [70]:
def matmul(A,B):
    A_rows, A_cols = A.shape
    B_rows, B_cols = B.shape
    assert A_cols==B_rows,\
    f"Inner dimensions must match: {A_cols} not equal to {B_rows}"
    C = torch.zeros([A_rows, B_cols])
    for i in range(A_rows):
        for j in range(B_cols):
            for k in range(A_cols):
                C[i,j] += A[i,k] * B[k,j]
    return C

In [71]:
matmul(a,b)

tensor([[ 6.,  6.],
        [10., 11.],
        [ 8., 13.]])

In [72]:
test_approximately(matmul(A, B), (A@B))

In [73]:
matmul_loop_time = timeit.timeit(partial(matmul,A,B), number=10)
matmul_loop_time

0.21083069499991325

In [74]:
# Call the matrix multiplication function
C = matmul(A, B)
print("Result Matrix C (A x B):")
print(C)

Result Matrix C (A x B):
tensor([[ -3.7884,   3.6057,   5.1554,   2.0474,   5.1080,   4.4804,  -2.9916,
          -2.6357,  -3.6968,   0.6317],
        [ -2.7225,  -1.8847,  -7.1586,   3.9262,  -2.7484,  -2.2764,   6.1097,
          -2.3526,  -2.7489,   5.6654],
        [ -1.8072,  -2.9585,   7.9122,   2.1853,   1.6055,   1.9326,  -4.1565,
          -2.6870,   2.0139,  -2.2262],
        [ -4.2416,  -3.3524,  -1.8086,   2.8147,   1.5821,   2.8664,   0.6469,
          -5.7378,  -5.1488,   2.8984],
        [  0.3146, -11.6253,  -2.3646,   4.5716,  -5.9525,   1.7613,   4.6121,
           5.6987,   3.2829,  -1.8484],
        [  6.1963,  -1.4186,   2.9474,  -2.1865,  -4.7232,  -0.3476,  -2.7012,
           3.6911,  -0.2264,  -0.3447],
        [ -1.7015,  -2.9590,   1.7325,   1.7626,   0.0950,   2.2935,  -1.4913,
          -1.6068,   2.3222,  -1.9108],
        [ -0.5190,  -7.8867,   0.3339,   2.4374,  -2.7916,   0.1827,   2.6353,
           8.0588,   2.7982,  -4.5685],
        [ -3.1114,  -0.

In [75]:
matmul_time = timeit.timeit(partial(matmul, A, B), number=1)
print(f"Time taken for matmul(A, B)L4GPU: {matmul_time:.6f} seconds")

Time taken for matmul(A, B)L4GPU: 0.021434 seconds
